# K-ABENA — Niveau 1 : CNN Vision (PyTorch)
K-ABENA filtre les **images déjà bien classifiées** avant la backward pass.
Les images difficiles restent toujours actives.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from kabena_ml.integrations.torch_utils import kabena_filter_torch

# Données CIFAR-10
tfm   = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)

# CNN simple
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8,256), nn.ReLU(), nn.Linear(256,10))
    def forward(self, x): return self.head(self.features(x))

model     = SmallCNN()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Boucle K-ABENA — 2 lignes de différence avec SGD standard
K, N = 0.30, 0.4
for epoch in range(5):
    total_gain, total_m, total_n = 0, 0, 0
    for X_b, y_b in loader:
        losses = F.cross_entropy(model(X_b), y_b, reduction='none')  # ← 'none'
        mask   = kabena_filter_torch(losses, K=K, N=N)               # ← +1 ligne
        if not mask.any(): continue

        optimizer.zero_grad()
        losses[mask].mean().backward()
        optimizer.step()

        m = mask.sum().item()
        total_m += m; total_n += len(y_b)

    gain = round((1 - total_m/total_n)*100)
    print(f"Epoch {epoch+1}/5 | gain moyen: {gain}% | {total_m}/{total_n} images actives")